# 지도 시각화

In [1]:
!pip install folium

### 기본 지도 그리기

In [2]:
import folium

m = folium.Map(
    location=[37.566826, 126.9786567],
    zoom_start=11
)

m.save('data/map_basic.html')
print('지도 저장 완료: ../data/map_basic.html')
m

지도 저장 완료: ../data/map_basic.html


In [3]:
import pandas as pd

marker = pd.read_excel('data/공공자전거 대여소 정보(25.12월 기준).xlsx', header=None)

marker = marker.iloc[5:100, [1, 4, 5]]
marker.columns = ['보관소명', '위도', '경도']
marker = marker.reset_index(drop = True)

marker

,보관소명,위도,경도
0,망원역 1번출구 앞,37.555649,126.910629
1,망원역 2번출구 앞,37.554951,126.910835
2,합정역 1번출구 앞,37.550629,126.914986
3,합정역 5번출구 앞,37.550007,126.914825
4,합정역 7번출구 앞,37.548645,126.912827
...,...,...,...
90,금융감독원 앞,37.523022,126.920837
91,여의도고교 앞,37.524837,126.934906
92,NH농협은행 앞,37.522079,126.930367
93,롯데캐슬엠파이어 옆,37.520695,126.925835


In [4]:
marker.isna().sum()

보관소명    0
위도      0
경도      0
dtype: int64

In [5]:
marker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   보관소명    95 non-null     object
 1   위도      95 non-null     object
 2   경도      95 non-null     object
dtypes: object(3)
memory usage: 2.4+ KB


In [6]:
# 위도, 경도 float type으로 변경
marker['위도'] = marker['위도'].astype(float)
marker['경도'] = marker['경도'].astype(float)

marker.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   보관소명    95 non-null     object 
 1   위도      95 non-null     float64
 2   경도      95 non-null     float64
dtypes: float64(2), object(1)
memory usage: 2.4+ KB


In [7]:
# 지도 생성
m_bike = folium.Map(location=[37.530, 126.990], zoom_start=12)

In [8]:
# 마커 생성
for idx, row in marker.iterrows():
    folium.Marker(
        location = [row['위도'], row['경도']],
        popup=folium.Popup(f"<b>{row['보관소명']}</b>", max_width=200),
        tooltip=row['보관소명'],
        icon=folium.Icon(color='blue', icon='info-sign')
    ).add_to(m_bike)

m_bike

In [37]:
df = pd.read_excel('data/공공자전거 대여소 정보(25.12월 기준).xlsx', header=None)
df = df.iloc[5:, [2, 4, 5, 8]].reset_index(drop = True)
df.columns = ['자치구', '위도', '경도', '거치대수']
df.head()

,자치구,위도,경도,거치대수
0,마포구,37.555649,126.910629,15
1,마포구,37.554951,126.910835,14
2,마포구,37.550629,126.914986,13
3,마포구,37.550007,126.914825,5
4,마포구,37.548645,126.912827,12


In [40]:
# 자치구 별 중심 위치, 거치대수 총합 => DF 생성

geo_bike = df.groupby('자치구').agg({
    '위도': 'mean',
    '경도': 'mean',
    '거치대수': 'sum'
}).reset_index()
geo_bike.columns = ['자치구', '위도', '경도', '거치대수합']
display(geo_bike)

,자치구,위도,경도,거치대수합
0,강남구,37.499357,127.057618,907
1,강동구,37.547701,127.147583,892
2,강북구,37.632387,127.025278,240
3,강서구,37.559022,126.835953,2129
4,관악구,37.479911,126.940712,354
5,광진구,37.544181,127.081756,721
6,구로구,37.493274,126.859816,608
7,금천구,37.466262,126.893385,400
8,노원구,37.64615,127.067944,1005
9,도봉구,37.660646,127.040351,468


In [47]:
m = folium.Map(location=[37.55, 126.98], zoom_start=11)

# 2. 데이터프레임의 행을 돌며 지도에 원(CircleMarker) 추가
for _, row in geo_bike.iterrows():
    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        # 💡 거치대수합 크기에 비례해서 반지름(radius) 설정 (값이 너무 크면 적당히 나눠주세요 예: row['거치대수합']/10)
        radius=row['거치대수합'] * 0.05, 
        popup=f"{row['자치구']}: {row['거치대수합']}대",
        color='blue',
        fill=True,
        fill_color='skyblue',
        fill_opacity=0.6
    ).add_to(m)

m